In [10]:
# Import dash dependencies
import dash
from dash.dependencies import Input, Output, State
import dash_bootstrap_components as dbc
from dash import dcc
from dash import html

# Import plotly dependencies
import plotly.express as px
import plotly.figure_factory as ff

# Import dependencies for io and data manipulation
import pandas as pd
import numpy as np
import math
import geopandas as gpd

# cos = pd.read_csv('data/input_csv.csv')
cos = pd.read_csv('data/csv_popdynamics_SSP2_2100.csv')
#  define earth radius (in meters) for hexagone resolution calculation)
earth_radius = 6.371e6
zoom=10
opacity=0.5

center_lat = cos[cos != 0.0].bfill(axis=1)['center_lat']
center_lon = cos[cos != 0.0].bfill(axis=1)['center_lon']
max= cos.loc[cos['popdynamics'].idxmax()]['popdynamics']
def roundup(x):
    return int(math.ceil(x / 10000)) * 10000
max=roundup(max)
label_max= int(math.ceil(max / 1000))
# print(label_max)

df_scenrio_year = pd.read_csv('data/BGD_Chittagong_popdynamics.csv')
df_scenrio_year["id"] = df_scenrio_year.index
df_scenrio_year = pd.wide_to_long(df_scenrio_year, stubnames='', i=['id'], j='year').reset_index()
df_scenrio_year.columns = [*df_scenrio_year.columns[:-1], 'Population']
ssps= df_scenrio_year['Scenario'].unique()
options=[{"value": x, "label": x} for x in ssps]
value= list(ssps) 

# Example map with OSM as basemap
url='data/geojson_popdynamics_SSP2_2100.geojson'
geo_df = gpd.read_file(url)

#################################

crp_logo = "assets/crp_logo.png" #path to crp logo

###################################################
#############  Start of the Dash app #############
###################################################
app = dash.Dash(__name__, 
                external_stylesheets=[dbc.themes.CERULEAN], # LUX FLATLY CERULEAN SPACELAB
                title='GFDRR.CRP  |  Future City Scan',
                meta_tags=[{'name': 'viewport',
                            'content': 'width=device-width, height=device-height, initial-scale=1, maximum-scale=1.2'}]
                )


###################################################
#############  Heroku #############
###################################################

server= app.server

###################################################
#############  Header /Navigation bar #############
###################################################
navbar = dbc.Navbar(
                [
                html.A(
                        # Use row and col to control vertical alignment of logo / brand
                        dbc.Row(
                                [   dbc.Col(html.Img(src=crp_logo, height="30px")),
                                    dbc.Col(dbc.NavbarBrand("   |  Future City Scan", className="ml-2")),
                                ],
                                align="center",
                                # no_gutters=True,
                                className="g-0",

                                ),
                        # href="https://www.gfdrr.org/en/crp",
                    ),
                ],
                color="dark",
                dark=True,
                className="mb-4",
)


###################################################
#############  Controls ##########################
###################################################

# Range Slider
targetdepth_tab = dbc.Card(
    dbc.CardBody(
        [
            html.H6("Select Population Range"),
            dcc.RangeSlider(
                            id='TargetDepth',
                            marks={
                                0: {'label': '0', 'style': {'fontSize': "0.6rem"}},
                                10000: {'label': '10k', 'style': {'fontSize': "0.6rem"}},
                                20000: {'label': '20k', 'style': {'fontSize': "0.6rem"}},
                                30000: {'label': '30k', 'style': {'fontSize': "0.6rem"}},
                                40000: {'label': '40k', 'style': {'fontSize': "0.6rem"}},
                                max: {'label': f'{label_max}k', 'style': {'fontSize': "0.6rem"}},
                                },
                            step=20,                # number of steps between values
                            min=0,
                            max=max,
                            value=[0,max],     # default value initially chosen4
                            dots=True,             # True, False - insert dots, only when step>1
                            allowCross=False,      # True,False - Manage handle crossover
                            disabled=False,        # True,False - disable handle
                            pushable=2,            # any number, or True with multiple handles
                            updatemode='mouseup',  # 'mouseup', 'drag' - update value method
                            included=True,         # True, False - highlight handle
                            vertical=False,        # True, False - vertical, horizontal slider
                            verticalHeight=900,    # hight of slider (pixels) when vertical=True
                            className='None',
                            tooltip={'always_visible':False,  # show current slider values
                                    'placement':'bottom'},
                            ),
        ]
    )
)

# Range Slider
targetdepth_tab_pop = dbc.Card(
    dbc.CardBody(
        [
            html.H6("Population Range"),
            dcc.RangeSlider(
                            id='TargetDepth_pop',
                            marks={
                                100: {'label': '0', 'style': {'fontSize': "0.6rem"}},
                                10000: {'label': '10k', 'style': {'fontSize': "0.6rem"}},
                                20000: {'label': '20k', 'style': {'fontSize': "0.6rem"}},
                                30000: {'label': '30k', 'style': {'fontSize': "0.6rem"}},
                                40000: {'label': '40k', 'style': {'fontSize': "0.6rem"}},
                                max: {'label': f'{label_max}k', 'style': {'fontSize': "0.6rem"}},
                                },
                            step=20,                # number of steps between values
                            min=100,
                            max=max,
                            value=[0,max],     # default value initially chosen4
                            dots=True,             # True, False - insert dots, only when step>1
                            allowCross=False,      # True,False - Manage handle crossover
                            disabled=False,        # True,False - disable handle
                            pushable=2,            # any number, or True with multiple handles
                            updatemode='mouseup',  # 'mouseup', 'drag' - update value method
                            included=True,         # True, False - highlight handle
                            vertical=False,        # True, False - vertical, horizontal slider
                            verticalHeight=900,    # hight of slider (pixels) when vertical=True
                            className='None',
                            tooltip={'always_visible':False,  # show current slider values
                                    'placement':'bottom'},
                            ),
        ]
    )
)


# Colormap choice and Hexbin resolution
colorscale_names = ['RdYlGn', 'Greys','RdBu','Viridis','Magma','Jet','IceFire']
control_tab_pop = dbc.CardGroup(
    [
        dbc.Card(
            dbc.CardBody(
                [
                    html.H6("Colormap"),
                    dcc.Dropdown(
                        id='colorscale_pop', 
                        options=[{"value": x, "label": x} 
                                for x in colorscale_names],
                        value='Jet',
                        style={'margin-bottom':'10px'}
                    ), 
                ]
            ),
         ),
    ]
)

# Colormap choice and Hexbin resolution
colorscale_names = ['RdYlGn', 'Greys','RdBu','Viridis','Magma','Jet','IceFire']
control_tab = dbc.CardGroup(
    [
        dbc.Card(
            dbc.CardBody(
                [
                    html.H6("Colormap"),
                    dcc.Dropdown(
                        id='colorscale', 
                        options=[{"value": x, "label": x} 
                                for x in colorscale_names],
                        value='RdYlGn',
                        style={'margin-bottom':'10px'}
                    ), 
                ]
            ),
         ),
        dbc.Card(   
            dbc.CardBody(
                [
                    html.H6("Hexbin resolution"),
                    dcc.Dropdown(
                        id="resolution",
                        options=[
                            {'label': '300 m', 'value': 300},
                            {'label': '500 m', 'value': 500},
                            {'label': '750 m', 'value': 750},
                            {'label': '1000 m', 'value': 1000},
                            {'label': '2000 m', 'value': 2000},
                            {'label': '3000 m', 'value': 3000},
                        ],
                        value=2000,
                        clearable=False
                    ) 
                ]
            )
        ),
    ]
)

credits_tab = dbc.Card(
    dbc.CardBody(
        dcc.Markdown(
            """
            Developed by CRP: [CRP.GFDRR](https://www.gfdrr.org/en/crp)
        
            """
        ),
    ),
    className="mt-0",
)


###################################################
#############  Controls 2 ##########################
###################################################

# Range Slider
targetdepth_tab2 = dbc.Card(
    dbc.CardBody(
        [
            html.H6("Control Target 2 "),
            dcc.RangeSlider(
                            id='TargetDepth2',
                            marks={
                                0: {'label': '0', 'style': {'fontSize': "0.6rem"}},
                                300: {'label': '300 m', 'style': {'fontSize': "0.6rem"}},     # key=position, value=what you see
                                500: {'label': '', 'style': {'fontSize': "0.6rem"}},
                                1000: {'label': '1000 m', 'style': {'fontSize': "0.6rem"}},
                                1500: {'label': '', 'style': {'fontSize': "0.6rem"}},
                                2000: {'label': '2000 m', 'style': {'fontSize': "0.6rem"}},
                                3000: {'label': '', 'style': {'fontSize': "0.6rem"}},
                                4000: {'label': '4000 m', 'style': {'fontSize': "0.6rem"}},
                                },
                            step=20,                # number of steps between values
                            min=0,
                            max=4000,
                            value=[0,4000],     # default value initially chosen4
                            dots=True,             # True, False - insert dots, only when step>1
                            allowCross=False,      # True,False - Manage handle crossover
                            disabled=False,        # True,False - disable handle
                            pushable=2,            # any number, or True with multiple handles
                            updatemode='mouseup',  # 'mouseup', 'drag' - update value method
                            included=True,         # True, False - highlight handle
                            vertical=False,        # True, False - vertical, horizontal slider
                            verticalHeight=900,    # hight of slider (pixels) when vertical=True
                            className='None',
                            tooltip={'always_visible':False,  # show current slider values
                                    'placement':'bottom'},
                            ),
        ]
    )
)


# Colormap choice and Hexbin resolution

colorscale_names = ['Greys','RdBu','Viridis','Magma','Jet','IceFire']

control_tab2 = dbc.CardGroup(
    [
        dbc.Card(
            dbc.CardBody(
                [
                    html.H6("Colormap 2"),
                    dcc.Dropdown(
                        id='colorscale2', 
                        options=[{"value": x, "label": x} 
                                for x in colorscale_names],
                        value='Viridis',
                        style={'margin-bottom':'10px'}
                    ), 
                ]
            ),
         ),
    ]
)

credits_tab2 = dbc.Card(
    dbc.CardBody(
        dcc.Markdown(
            """
            This map estimates population numbers per 10,000 meter sq. grid cell. It provides a more consistent representation of population distributions across different landscapes than administrative unit counts. 
            
            """
        ),
    ),
    className="mt-02",
)


population_card = dbc.Card(
    dbc.CardBody(
        dcc.Markdown(
            """
            Statiscal or hot spot analysis shows: insert key message----->> This map estimates population numbers per 10,000 meter sq. grid cell. It provides a more consistent representation of population distributions across different landscapes than administrative unit counts. 
            
            """
        ),
    ),
    className="mt-03",
)

# Left side controls for the scatterplot
control_tab3 = dbc.CardGroup(
    [
        dbc.Card(
            dbc.CardBody(
                [
                    html.H6("Colormap 3"),
                    dcc.Dropdown(
                        id='colorscale3', 
                        options=[{"value": x, "label": x} 
                                for x in colorscale_names],
                        value='Magma',
                        style={'margin-bottom':'10px'}
                    ), 
                ]
            ),
         ),
    ]
)

credits_tab3 = dbc.Card(
    dbc.CardBody(
        dcc.Markdown(
            """
            The map illustrates monthly temporal changes from 2014 to 2022 in the emission of nighttime lights, indicating changes in economic activity. Positive values represent an increase in the intensity of nighttime light emission and, by proxy, economic activity.

            """
        ),
    ),
    className="mt-03",
)

# Add only description card for the fourth scatterplot
credits_tab4 = dbc.Card(
    dbc.CardBody(
        dcc.Markdown(
            """
            GDP growth measures how fast the local economy is growing. Positive GDP growth is typically attributed to government spending, personal consumption, business investment, construction, and net trade.
            """
        ),
    ),
    className="mt-04",
)

###################################################
#############  Right layout (=Maps) ###############
###################################################

data_density_map_component = dbc.Card(
    [
        dbc.CardHeader(
            html.H3("Section 1: Demographic Distribution")), 
            dbc.CardBody(
                [
                    dcc.Markdown(
                        """

                        The City Resilience Program (CRP) works to build resilient cities with the capacity to plan for and mitigate adverse impacts of disasters and climate change, thus enabling them to save lives, reduce losses, and unlock economic and social potential. The program is a partnership between the World Bank and GFDRR. A City Scan is a rapid geospatial assessment of the critical resilience challenges that cities face using the best publicly available global datasets and open source tools. The output is a package of maps, data visualizations and insights that integrate features of both the built and natural environments. 
                        
                        """
                    ),
                    dcc.Graph(
                        id='data_density' , 
                        # style={'height': 800},
                        ), 
                ],
                style={'padding':'0.5'}
            ) ,
            population_card
            , 
            html.H6("Section: Another map"), 
                    dcc.Graph(
                        id='data_density_' , 
                        style={'padding':'0.5'},
                        ),  
            html.Br()  ,
            html.Br()  ,          
            html.Br()  ,          
            html.H5('Demographic Projections'),
            dcc.Checklist(
                id="checklist",
                options= options, 
                value= value  , 
                inline=True
            ),    
            dcc.Graph(id="ssps_graph",),
    ], 
    className="my-2", 
    style={'height': 2000 , 'padding':'0.5'}
)

data_cos_component = dbc.Card(
    [
        dbc.CardHeader(
            html.H3("Section 2: GDP Distribution")), 
            dbc.CardBody(
                [
                    dcc.Markdown(
                        """
                        The authors are longstanding collaborators of Brian O’Neill, who is one of the SSPs developers. The authors downscale 1/8 degree maps into 1km maps based on the 1-km total population count map in 2000 of the Global Rural-Urban Mapping Project (GRUMP 2000), so might be a bit outdated. 

                        """
                    ),
                    dcc.Graph(
                        id='data_cos' , 
                        style={'height': 800},
                        # responsive=True
                        ), 
                ],
                style={'padding':'0.5'}
            )
    ], 
    className="my-2", 
    style={'height': 1600}
)



fig_tab = dbc.Tabs(
    [
        dbc.Tab(data_density_map_component, label="Section 1: Demographic Distribution",active_label_style={"font-weight":"800","color": "#00AEF9"}),
        dbc.Tab(data_cos_component, label="Section 2: GDP Distribution",active_label_style={"font-weight":"800","color": "#00AEF9"}),

    ]
)

###################################
######  Layout of Dash app ########
###################################

app.layout = dbc.Container(
    [
        navbar,
        dbc.Row(
            [
                dbc.Col(  # left layout
                    [
                    dbc.CardHeader("Credits"),
                    credits_tab,
                    html.Label('', style={'paddingTop': '15rem'}),
                    dbc.Row(
                            [
                            dbc.CardHeader("Data control 2"),
                            targetdepth_tab, 
                            control_tab,
                            ], 
                            ),
                    html.Label('', style={'paddingTop': '25rem'}),
                    dbc.Row(
                            [
                            dbc.CardHeader("Population Slider and Color"),
                            targetdepth_tab_pop, 
                            control_tab_pop,
                            ], 
                            ),
                    ], 
                    #
                        width=4),

                dbc.Col( # right layout
                    [
                    fig_tab,
                    ],
                    width=8,
                ),
            ]
        ),
    ],

    fluid=True,
)

###################################
######  Section 1: Demographic Distribution ########
###################################

@app.callback(
    Output('data_density', 'figure'),
    [
     Input('resolution', 'value'),
     Input('TargetDepth', 'value'),
     Input("colorscale", "value"),
    ]
)
def update_data_density(resolution, depth, scale):

    dff= cos.copy()
    
    # dff = dff[(dff["popdynamics"] >= depth[0]) & (dff["popdynamics"] <= depth[1])]
    dff = dff[(dff["popdynamics"] >= depth[0]) & (dff["popdynamics"] <= depth[1])]
    heg_size = -1* ((dff.lat.max() - dff.lat.min()) / resolution * np.pi / 180 * earth_radius  * np.cos(dff.lon.mean()))
    heg_size = math.floor(heg_size)  # define hegsize dimension
    

    fig_data_density = ff.create_hexbin_mapbox(data_frame=dff, 
                                        lat="lat", lon="lon",
                                        nx_hexagon=heg_size, 
                                        opacity=opacity, 
                                        labels={"color": "Density index"},
                                        color="popdynamics", 
                                        agg_func=np.sum, 
                                        mapbox_style='carto-positron',
                                        color_continuous_scale=scale,
                                        show_original_data=False, 
                                        original_data_marker=dict(opacity=0.6, size=4, color="black"),
                                        min_count=1,
                                        zoom=10,
                                        center= {"lon": center_lon[0], "lat": center_lat[0]}
                                        )

    fig_data_density.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0},
                    
                            showlegend=False,
                            coloraxis_showscale=False,  
                            hoverlabel=dict(
                                bgcolor="#3a3a3b",
                                font_color='white',
                                # color='white',
                                font_size=16,
                                font_family="Nunito Sans"
                            )
                         )
    fig_data_density.data[0].hovertemplate = "<span style='font-size:1.2rem; font-weight=400'>Number of People = %{z:,.0f}</span><br><br>"

    return fig_data_density

############################################
#reapteat section 1 


@app.callback(
    Output('data_density_', 'figure'),
    [
     Input('TargetDepth_pop', 'value'),
     Input("colorscale_pop", "value"),
    ]
)
def update_fig_data_density_( depth, scale):
    dff= geo_df.copy()
    dff = dff[(dff["popdynamics"] >= depth[0]) & (dff["popdynamics"] <= depth[1])]    
    fig_data_density_ = px.choropleth_mapbox(dff,
                           geojson=dff.geometry,
                           locations=dff.index,
                           color="popdynamics",
                           color_continuous_scale=scale,
                           center={"lat": center_lat[0], "lon": center_lon[0]},
                           mapbox_style="open-street-map",
                           zoom=10,
                           )
    fig_data_density_.update_traces(marker_line_width=0)
    fig_data_density_.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
    fig_data_density_.data[0].hovertemplate = "<span style='font-size:1.2rem; font-weight=400'>Population = %{z:,.0f}</span><br><br>"

    return fig_data_density_

############################################
############################################

@app.callback(
    Output("ssps_graph", "figure"), 
    Input("checklist", "value"))
def update_line_chart(Scenario):
    # df = px.data.gapminder() # replace with your own data source
    mask = df_scenrio_year.Scenario.isin(Scenario)
    line_chart = px.line(df_scenrio_year[mask], 
        x="year", y="Population", color='Scenario')
    return line_chart

############################################
#####Section 2 GDP-->>>Interactive  ########
############################################


@app.callback(
    Output('data_cos', 'figure'),
    [
     Input('resolution', 'value'),
     Input('TargetDepth', 'value'),
     Input("colorscale", "value"),
    ]
)
def update_data_cos(resolution, depth, scale):

    dff= cos.copy()
    dff = dff[(dff["popdynamics"] >= depth[0]) & (dff["popdynamics"] <= depth[1])]
    heg_size = -1* ((dff.lat.max() - dff.lat.min()) / resolution * np.pi / 180 * earth_radius  * np.cos(dff.lon.mean()))
    heg_size = math.floor(heg_size)  # define hegsize dimension
    

    fig_cos = ff.create_hexbin_mapbox(data_frame=dff, 
                                        lat="lat", lon="lon",
                                        nx_hexagon=heg_size, 
                                        opacity=opacity, 
                                        labels={"color": "Density index"},
                                        color="popdynamics", 
                                        agg_func=np.sum, 
                                        mapbox_style='carto-positron',
                                        color_continuous_scale=scale,
                                        show_original_data=False, 
                                        original_data_marker=dict(opacity=0.6, size=4, color="black"),
                                        min_count=1,
                                        zoom=10,
                                        center= {"lon": center_lon[0], "lat": center_lat[0]}
                                        )
    
    fig_cos.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0},
                    
                            showlegend=False,
                            coloraxis_showscale=False,  
                            hoverlabel=dict(
                                bgcolor="#3a3a3b",
                                font_color='white',
                                # color='white',
                                font_size=16,
                                font_family="Nunito Sans"
                            )
                         )
    fig_cos.data[0].hovertemplate = "<span style='font-size:1.2rem; font-weight=400'>COS index = %{z:,.0f}</span><br><br>"

    return fig_cos


if __name__ == "__main__":
    app.run_server(debug=True)
    # app.run_server()
